# Exploração do Dataset Paderborn (PU Dataset) e Escalogramas CWT

**Projeto:** Diagnóstico de Falhas em Rolamentos Industriais via CWT + Deep Learning  
**Dataset:** Paderborn University (KAt) Bearing Data Center  
**Objetivo:** Explorar os sinais de vibração de alta resolução (**64 kHz**) do dataset de Paderborn, analisando falhas reais induzidas por ensaios acelerados de fadiga (*fatigue flaking*) e gerando escalogramas bidimensionais via Transformada Wavelet Contínua (CWT) com wavelet de Morlet.

---

## 1. Importação dos Módulos e Parâmetros

Importamos as bibliotecas e os módulos dedicados da pasta `src/`.

In [ ]:
import sys
from pathlib import Path

# Adicionar raiz do projeto ao sys.path
BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import numpy as np
import matplotlib.pyplot as plt

from src.config import (
    PADERBORN_RAW_DIR,
    PADERBORN_PROCESSED_DIR,
    PADERBORN_CLASSES,
    PADERBORN_FS,
    PADERBORN_WINDOW_SIZE,
    PADERBORN_FREQ_MIN,
    PADERBORN_FREQ_MAX
)
from src.dataset_paderborn import load_paderborn_mat_file, segment_paderborn_signal
from src.generate_dataset_paderborn import compute_paderborn_cwt
from src.visualization import plot_monograph_figure

## 2. Resumo das Configurações do Dataset Paderborn

Exibimos os parâmetros de amostragem de alta fidelidade e configurações das janelas:

In [ ]:
print("=" * 60)
print(" CONFIGURAÇÕES DO DATASET PADERBORN (PU DATASET) ")
print("=" * 60)
print(f"Taxa de Amostragem (Fs)  : {PADERBORN_FS:,} Hz (64 kHz)")
print(f"Comprimento da Janela     : {PADERBORN_WINDOW_SIZE} amostras (~{PADERBORN_WINDOW_SIZE/PADERBORN_FS*1000:.1f} ms)")
print(f"Faixa de Frequência CWT   : {PADERBORN_FREQ_MIN} Hz a {PADERBORN_FREQ_MAX:,} Hz")
print(f"Classes Mapeadas          : {PADERBORN_CLASSES}")
print("=" * 60)

## 3. Carregamento e Comparação dos Sinais de Vibração (64 kHz)

Carregamos um sinal de cada condição: **Saudável (`K001`)**, **Pista Interna (`KI14`)** e **Pista Externa (`KA15`)**.

In [ ]:
sample_k001 = list((PADERBORN_RAW_DIR / 'K001').glob('*.mat'))[0]
sample_ki14 = list((PADERBORN_RAW_DIR / 'KI14').glob('*.mat'))[0]
sample_ka15 = list((PADERBORN_RAW_DIR / 'KA15').glob('*.mat'))[0]

sig_normal = load_paderborn_mat_file(sample_k001)
sig_inner  = load_paderborn_mat_file(sample_ki14)
sig_outer  = load_paderborn_mat_file(sample_ka15)

print(f"[+] K001 (Normal)     : {len(sig_normal):,} amostras ({sample_k001.name})")
print(f"[+] KI14 (Inner Race) : {len(sig_inner):,} amostras ({sample_ki14.name})")
print(f"[+] KA15 (Outer Race) : {len(sig_outer):,} amostras ({sample_ka15.name})")

## 4. Visualização dos Sinais no Domínio do Tempo e Escalogramas CWT

Plotamos a comparação visual no padrão acadêmico para as 3 condições:

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
t = np.arange(PADERBORN_WINDOW_SIZE) / PADERBORN_FS

axes[0].plot(t, sig_normal[:PADERBORN_WINDOW_SIZE], color='#2ca02c', linewidth=0.8)
axes[0].set_title("Paderborn: Rolamento Saudável (K001 - Normal)", fontweight="bold")
axes[0].set_ylabel("Aceleração (g)")
axes[0].grid(True, linestyle="--", alpha=0.5)

axes[1].plot(t, sig_inner[:PADERBORN_WINDOW_SIZE], color='#1f77b4', linewidth=0.8)
axes[1].set_title("Paderborn: Falha Real na Pista Interna (KI14 - Inner Race Fatigue)", fontweight="bold")
axes[1].set_ylabel("Aceleração (g)")
axes[1].grid(True, linestyle="--", alpha=0.5)

axes[2].plot(t, sig_outer[:PADERBORN_WINDOW_SIZE], color='#d62728', linewidth=0.8)
axes[2].set_title("Paderborn: Falha Real na Pista Externa (KA15 - Outer Race Fatigue)", fontweight="bold")
axes[2].set_ylabel("Aceleração (g)")
axes[2].set_xlabel("Tempo (s)")
axes[2].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 5. Escalogramas CWT Individuais

Geramos a representação tempo-frequência CWT para uma janela com falha real na pista interna:

In [ ]:
janela_inner = sig_inner[:PADERBORN_WINDOW_SIZE]
cwt_mat = compute_paderborn_cwt(janela_inner)
t_win = np.arange(len(janela_inner)) / PADERBORN_FS

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(
    cwt_mat,
    extent=[t_win[0], t_win[-1], PADERBORN_FREQ_MIN, PADERBORN_FREQ_MAX],
    cmap="jet",
    aspect="auto",
    origin="lower"
)
ax.set_title("Escalograma CWT (Morlet) — Paderborn KI14 (Pista Interna)", fontsize=12, fontweight="bold")
ax.set_xlabel("Tempo (s)")
ax.set_ylabel("Frequência (Hz)")
plt.colorbar(im, ax=ax, label="Magnitude CWT")
plt.tight_layout()
plt.show()

## 6. Geração do Dataset Completo de Escalogramas CWT para Treinamento

Executamos o pipeline automatizado do `src/generate_dataset_paderborn.py`.

> **Dica:** Você pode passar `max_files_per_class=10` para uma geração ultrarrápida de teste, ou deixar `None` para processar todos os arquivos.

In [ ]:
from src.generate_dataset_paderborn import process_and_generate_paderborn_dataset

# Para processar todos os arquivos, use max_files_per_class=None
# Para um teste inicial rápido, use max_files_per_class=10
process_and_generate_paderborn_dataset(max_files_per_class=10)